# STAGE B/5 — LLM 32B gán SILVER trên 100 file thật (+ synth)

**Settings:** GPU **T4 x2** (cần 2 GPU cho tensor-parallel) + Internet ON. **~1.5–2.5h**.
Tải Qwen2.5-32B-AWQ (~19GB). Xong → **Commit**. Stage C *Add Input* để lấy `silver.jsonl`+`llm.jsonl`.

⚠️ Bước cài vLLM là chỗ DỄ VỠ nhất (pin version). Nếu lỗi version, đọc cell deps.


In [ ]:
# Clone repo
import os, subprocess
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
# Deps + recipe vLLM 32B-AWQ trên 2xT4 (V0 + XFORMERS, KHÔNG flashinfer)
!pip install -q rapidfuzz pyyaml regex
!pip install -q "vllm==0.8.5" "transformers==4.51.3"
# FIX lỗi 'runtime_version': pin vllm kéo protobuf về 4.25 (thiếu) -> ép lại >=5.26.
#   Đây là LỖI Stage B lần trước. Chạy SAU cùng để đè bản protobuf cũ.
!pip install -q "protobuf>=5.26,<6"
import os
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"
print("vLLM env set (V0 + XFORMERS)")


In [ ]:
# Gán silver (5 mẫu/file, vote>=2) + sinh 2000 note synth — cùng 1 lần load model
!python scripts/gen_silver.py --model Qwen/Qwen2.5-32B-Instruct-AWQ \
   --n_samples 5 --min_votes 2 --n_synth 2000 --tp 2 \
   --out data/silver.jsonl --synth_out data/synthetic/llm.jsonl


In [ ]:
# Lưu ra /kaggle/working để Commit
import shutil, os
for f in ["data/silver.jsonl","data/synthetic/llm.jsonl"]:
    if os.path.exists(f): shutil.copy(f, "/kaggle/working/"+os.path.basename(f)); print("SAVED", f)
print("Commit roi qua Stage C")


**Ghi chú:** silver = data quan trọng nhất (đúng miền thật). Nếu vLLM OOM: giảm `--n_samples 3` hoặc `--max_model_len 4096`. Nếu quá giờ: `--n_synth 0` (chỉ silver).
